# Stability — feature-importance distance, parameter distance

Bootstraps each model to see how much its feature importances, parameters, and AUC move
under resampled training data, for whichever models have a `models/<name>_model.py` file
so far. Bootstrapping means refitting `N_BOOT` times, so compute cost is the main
constraint here — see the per-model `N_BOOT`/`STABILITY_SAMPLE_SIZE` further down.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("../..").resolve()))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from common_metrics import (
    FEATURES, MODEL_NAMES, TEAM_THRESHOLD,
    load_split, get_X_y, available_models, report_status, importance_vector, get_or_fit_model, run_step,
)

OUTPUT_DIR = Path("../../outputs/evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report_status()

## Load data

In [ ]:
train_df = load_split("train")
test_df = load_split("test")

X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

print(f"train: {X_train.shape}, test: {X_test.shape}")

## Feature-importance distance + performance stability — one bootstrap, both numbers

`d(f1, f2) = ||φ(f1) - φ(f2)||₂`, via `common_metrics.importance_vector` so it's
model-agnostic and comparable across xgboost/logreg/tabpfn. AUC spread is bundled into the
same loop since both need the same bootstrap resamples — one fit per iteration gets you both
numbers instead of fitting twice.

TabPFN's `fit()` redraws its ~10K-row context each time, so bootstrapping it tests
sensitivity to *which* context gets sampled, not just ordinary resampling noise — arguably
more informative at this dataset's scale. If TabPFN's context turns out to be fixed instead,
this reduces to plain sampling noise like the other two models.

TabPFN's CPU cost hits every `predict_proba` call, not just `fit()` — the permutation-importance
fallback alone is ~150 calls per iteration, plus one full pass over `X_test` for AUC, so cost
scales with row count. `sample_size` subsamples `X_train`/`X_test` once per model so everything
downstream runs smaller — a coarser stability estimate for whichever model can't afford full size
(`STABILITY_SAMPLE_SIZE` below).

In [ ]:
def stability_bootstrap(name, module, model_type, X_train, y_train, X_test, y_test, n_boot=30, sample_size=None, random_state=42):
    rng = np.random.RandomState(random_state)
    if sample_size is not None:
        train_idx = rng.choice(len(X_train), size=min(sample_size, len(X_train)), replace=False)
        X_train, y_train = X_train.iloc[train_idx], y_train.iloc[train_idx]
        test_idx = rng.choice(len(X_test), size=min(sample_size, len(X_test)), replace=False)
        X_test, y_test = X_test.iloc[test_idx], y_test.iloc[test_idx]

    base_model = get_or_fit_model(name, module, X_train, y_train)  # the real model if saved, else a fresh fit
    base_imp = importance_vector(model_type, base_model, module, X_train, y_train)

    imp_distances, aucs = [], []
    for i in range(n_boot):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        X_b, y_b = X_train.iloc[idx], y_train.iloc[idx]
        model_b = module.fit(X_b, y_b)  # resampled data has no saved artifact — must fit
        imp_b = importance_vector(model_type, model_b, module, X_b, y_b)
        imp_distances.append(float(np.linalg.norm(base_imp - imp_b)))
        probs = module.predict_proba(model_b, X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, probs))
    return imp_distances, aucs

## Parameter distance — logreg only

In [ ]:
def parameter_distance_bootstrap(name, module, X_train, y_train, n_boot=30, random_state=42):
    rng = np.random.RandomState(random_state)
    base_model = get_or_fit_model(name, module, X_train, y_train)
    theta_base = np.asarray(base_model.coef_[0])
    distances = []
    for i in range(n_boot):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        X_b, y_b = X_train.iloc[idx], y_train.iloc[idx]
        theta_b = np.asarray(module.fit(X_b, y_b).coef_[0])
        distances.append(float(np.linalg.norm(theta_base - theta_b)))
    return distances

## D1 → D2 — has the model converged, or would more data still change it?

Syllabus §7.1's worked example: fit on D1 (50% of train) and D2 (100%), compare feature
importances directly, no resampling. Different question from the bootstrap above — that
asks how much the model wobbles from sampling noise at a fixed size, this asks whether it's
still moving as data grows. Only 2 fits, so it's cheap.

If TabPFN's `fit()` subsamples to a fixed context size regardless of how much data it's
given, D1 and D2 could collapse to the same effective context — worth checking if this
number comes back suspiciously small for TabPFN.

In [ ]:
def stability_d1_d2(name, module, model_type, X_train, y_train, frac_d1=0.5, sample_size=None, random_state=42):
    rng = np.random.RandomState(random_state)
    if sample_size is not None:
        idx = rng.choice(len(X_train), size=min(sample_size, len(X_train)), replace=False)
        X_train, y_train = X_train.iloc[idx], y_train.iloc[idx]

    idx_d1 = rng.choice(len(X_train), size=int(len(X_train) * frac_d1), replace=False)
    X_d1, y_d1 = X_train.iloc[idx_d1], y_train.iloc[idx_d1]

    model_d1 = module.fit(X_d1, y_d1)  # D1 (50%) has no saved artifact — must fit
    model_d2 = get_or_fit_model(name, module, X_train, y_train)  # D2 (100%) is exactly what the saved model was trained on

    imp_d1 = importance_vector(model_type, model_d1, module, X_d1, y_d1)
    imp_d2 = importance_vector(model_type, model_d2, module, X_train, y_train)
    return float(np.linalg.norm(imp_d1 - imp_d2))

## Run across available models

`N_BOOT`/`STABILITY_SAMPLE_SIZE` are per-model. XGBoost and logreg fit in under a second, so
they get the full 30-iteration bootstrap on full data. TabPFN's CPU inference (≈0.375s/row)
can't afford that, so it runs fewer iterations on a 5,000-row subsample — coarser, not a
different metric. Bootstrap and D1→D2 each go through `run_step`, so one failing doesn't
lose the other.

**Caching**: same idea as the interpretability notebook — if `outputs/evaluation/stability_results.pkl`
exists and covers the same models currently available, this loop is skipped and
`stability_results` loads from disk instead, so changing a plot below doesn't re-run every
bootstrap. Set `FORCE_RECOMPUTE = True` to ignore it.

In [ ]:
MODEL_TYPE = {"xgboost": "xgboost", "logreg": "logreg", "tabpfn": "tabpfn"}
N_BOOT = {"xgboost": 30, "logreg": 30, "tabpfn": 10}
STABILITY_SAMPLE_SIZE = {"tabpfn": 5000}  # None (full data) for xgboost/logreg
FORCE_RECOMPUTE = False  # set True to ignore the cache below and recompute everything
RESULTS_CACHE = OUTPUT_DIR / "stability_results.pkl"

stability_results = None
if RESULTS_CACHE.exists() and not FORCE_RECOMPUTE:
    with open(RESULTS_CACHE, "rb") as f:
        cached = pickle.load(f)
    if set(cached.keys()) == set(available_models().keys()):
        stability_results = cached
        print(f"Loaded cached results for {list(stability_results.keys())} from {RESULTS_CACHE}")
        print("Set FORCE_RECOMPUTE = True above and rerun this cell to recompute from scratch.")
    else:
        print(f"Cache has {list(cached.keys())}, available models are {list(available_models().keys())} — recomputing")

if stability_results is None:
    stability_results = {}
    for name, module in available_models().items():
        print(f"\n=== {name} ===")
        model_type = MODEL_TYPE[name]
        n_boot = N_BOOT.get(name, 30)
        sample_size = STABILITY_SAMPLE_SIZE.get(name)

        entry = {}
        if run_step(entry, "_bootstrap", stability_bootstrap, name, module, model_type, X_train, y_train, X_test, y_test, n_boot=n_boot, sample_size=sample_size):
            entry["importance_distance"], entry["auc_spread"] = entry.pop("_bootstrap")
            print(f"  importance distance (bootstrap, n={n_boot}): mean={np.mean(entry['importance_distance']):.4f}, std={np.std(entry['importance_distance']):.4f}")
            print(f"  AUC spread: mean={np.mean(entry['auc_spread']):.4f}, std={np.std(entry['auc_spread']):.4f}")

        if run_step(entry, "d1_d2_distance", stability_d1_d2, name, module, model_type, X_train, y_train, sample_size=sample_size):
            print(f"  D1→D2 importance distance (50% → 100% of data): {entry['d1_d2_distance']:.4f}")

        if name == "logreg":
            if run_step(entry, "parameter_distance", parameter_distance_bootstrap, name, module, X_train, y_train, n_boot=n_boot):
                print(f"  parameter distance: mean={np.mean(entry['parameter_distance']):.4f}, std={np.std(entry['parameter_distance']):.4f}")

        if entry:
            stability_results[name] = entry

if not stability_results:
    print("No models ready yet — drop a models/<name>_model.py file in and re-run.")

## Plot — importance-distance distribution per model

In [ ]:
plottable = {name: r["importance_distance"] for name, r in stability_results.items() if "importance_distance" in r}
if plottable:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot(list(plottable.values()), tick_labels=list(plottable.keys()))
    ax.set_ylabel("feature-importance distance (bootstrap)")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "stability_importance_distance_boxplot.png", dpi=150, bbox_inches="tight")
    plt.show()

## Summary table — one row per model, copy-ready for the slide deck

In [ ]:
summary_rows = []
for name, r in stability_results.items():
    summary_rows.append({
        "model": name,
        "n_boot": N_BOOT.get(name, 30),
        "importance distance (mean)": round(np.mean(r["importance_distance"]), 4) if "importance_distance" in r else "—",
        "AUC spread (std)": round(np.std(r["auc_spread"]), 4) if "auc_spread" in r else "—",
        "D1→D2 distance": round(r["d1_d2_distance"], 4) if "d1_d2_distance" in r else "—",
        "parameter distance (mean)": round(np.mean(r["parameter_distance"]), 4) if "parameter_distance" in r else "—",
    })

stability_summary_df = pd.DataFrame(summary_rows)
stability_summary_df

## Smoke test — remove once real models are in `models/`

Same throwaway logistic regression as the interpretability notebook, on a small
sample and few bootstrap iterations, purely to check the harness runs.

In [ ]:
from sklearn.linear_model import LogisticRegression

class _SmokeTestModule:
    _medians = None  # fixed at fit time so a later all-NaN batch (e.g. a masked coalition) still fills

    @staticmethod
    def fit(X, y):
        Xn = X.select_dtypes("number")
        _SmokeTestModule._medians = Xn.median().fillna(0)
        return LogisticRegression(max_iter=200).fit(Xn.fillna(_SmokeTestModule._medians), y)

    @staticmethod
    def predict_proba(model, X):
        Xn = X.select_dtypes("number").fillna(_SmokeTestModule._medians)
        return model.predict_proba(Xn)

if not stability_results:
    print("Running a throwaway smoke test — NOT a real model, just checking the harness works end to end.")
    sample = train_df.sample(20_000, random_state=42)
    Xs, ys = get_X_y(sample)
    test_sample = test_df.sample(2_000, random_state=42)
    Xt, yt = get_X_y(test_sample)
    smoke_imp, smoke_auc = stability_bootstrap("logreg", _SmokeTestModule, "logreg", Xs, ys, Xt, yt, n_boot=5)
    print("Smoke test importance distances:", smoke_imp, "— harness is wired correctly.")

## Save outputs

Writes to `outputs/evaluation/` (gitignored, same as `models/` and `data/`) so results survive
independently of the notebook. Everything here is already lightweight numbers (distances, AUCs),
no model objects, so the whole `stability_results` dict is pickled as-is.

In [ ]:
if stability_results:
    stability_summary_df.to_csv(OUTPUT_DIR / "stability_summary.csv", index=False)
    with open(RESULTS_CACHE, "wb") as f:
        pickle.dump(stability_results, f)
    print(f"Saved summary + full results to {OUTPUT_DIR}/")